# FarmRoute Lite Training

Trains the crop disease classifier used by the FarmRoute Android app and exports the two files the app loads from `app/src/main/assets/`:

| File | What it is |
|------|------------|
| `farmroute_disease_model.tflite` | MobileNetV2 (width 0.35, 160 x 160 input), int8 quantised, float input and output |
| `labels.txt` | One class name per line, in the order of the model's outputs |

**Run it in Google Colab with a GPU:** Runtime > Change runtime type > T4 GPU, then Runtime > Run all. On the free tier the whole run takes roughly 30 to 45 minutes, most of it downloading images.

The model is kept small so it runs on entry level phones: about 0.6 MB on disk, a few tens of milliseconds per image on a budget CPU.

### Contract with the app

`TFLiteClassifier.java` makes three assumptions. This notebook keeps all three, and the export section checks them.

1. **Input is 160 x 160 RGB float32, scaled to [-1, 1]** (`pixel / 127.5 - 1`). The app does this scaling itself, so the model must *not* contain its own preprocessing layer. If you change `IMG_SIZE` here, change `INPUT_SIZE` in `TFLiteClassifier.java` too.
2. **Output is one float32 probability per class** (softmax), in the same order as `labels.txt`.
3. **The model loads on TensorFlow Lite 2.14**, the runtime version in `app/build.gradle`.

## 1. Configuration

In [ ]:
import os

# Crops most relevant to Zimbabwean smallholders: maize, tomato, potato, pepper.
# Names match the PlantVillage folder names exactly. Order here is the order of
# the model's outputs and of labels.txt.
CLASSES = [
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot",
    "Corn_(maize)___Common_rust_",
    "Corn_(maize)___Northern_Leaf_Blight",
    "Corn_(maize)___healthy",
    "Pepper,_bell___Bacterial_spot",
    "Pepper,_bell___healthy",
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
    "Tomato___Bacterial_spot",
    "Tomato___Early_blight",
    "Tomato___Late_blight",
    "Tomato___Leaf_Mold",
    "Tomato___Septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite",
    "Tomato___Target_Spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus",
    "Tomato___Tomato_mosaic_virus",
    "Tomato___healthy",
]

IMG_SIZE = 160   # Must equal INPUT_SIZE in TFLiteClassifier.java.
ALPHA = 0.35     # MobileNetV2 width multiplier. 0.35 is the smallest with ImageNet weights.
BATCH_SIZE = 32
EPOCHS_HEAD = 8          # Stage 1: train only the new classification head.
EPOCHS_FINE_TUNE = 8     # Stage 2: also train the top of the backbone.
FINE_TUNE_LAYERS = 40    # How many backbone layers (from the top) to unfreeze in stage 2.
SEED = 42

WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
DATASET_ROOT = os.path.join(WORK_DIR, "PlantVillage-Dataset")
# Folder that directly contains one sub-folder per class. If you already have
# PlantVillage (for example the Kaggle copy, whose folder is
# "plantvillage dataset/color"), point DATA_DIR at it and the download is skipped.
DATA_DIR = os.path.join(DATASET_ROOT, "raw", "color")
OUTPUT_DIR = os.path.join(WORK_DIR, "farmroute_export")

MODEL_FILE = "farmroute_disease_model.tflite"   # Name TFLiteClassifier.java loads.
LABELS_FILE = "labels.txt"

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
import keras

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none (training will be slow)")

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Get the images

Downloads only the 19 needed class folders of the [PlantVillage dataset](https://github.com/spMohanty/PlantVillage-Dataset) (about 26,000 colour leaf photos) with a sparse git checkout, instead of the whole multi-gigabyte repository. Skipped if `DATA_DIR` already holds the class folders.

In [ ]:
# DOWNLOAD
import subprocess

def have_all_classes(root):
    return all(os.path.isdir(os.path.join(root, c)) for c in CLASSES)

if have_all_classes(DATA_DIR):
    print("Dataset already present at", DATA_DIR)
else:
    if not os.path.isdir(DATASET_ROOT):
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                        "https://github.com/spMohanty/PlantVillage-Dataset.git", DATASET_ROOT],
                       check=True)
    subprocess.run(["git", "-C", DATASET_ROOT, "sparse-checkout", "set"]
                   + [f"raw/color/{c}" for c in CLASSES], check=True)
    print("Downloaded to", DATA_DIR)

In [ ]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

missing = [c for c in CLASSES if not os.path.isdir(os.path.join(DATA_DIR, c))]
if missing:
    raise FileNotFoundError(f"DATA_DIR {DATA_DIR} is missing class folders: {missing}")

files_by_class = {}
for c in CLASSES:
    folder = os.path.join(DATA_DIR, c)
    files_by_class[c] = sorted(
        os.path.join(folder, f) for f in os.listdir(folder)
        if f.lower().endswith(IMAGE_EXTENSIONS))

counts = {c: len(f) for c, f in files_by_class.items()}
print(f"{sum(counts.values())} images in {len(CLASSES)} classes\n")
for c, n in counts.items():
    print(f"{n:6d}  {c}")

## 3. Split and build the input pipeline

Each class is split 70 / 15 / 15 into train, validation and test on its own, so small classes such as healthy potato are represented in every split. Images are streamed from disk batch by batch rather than loaded into memory, which keeps the notebook within free tier RAM.

Training images get random flips, 90 degree rotations, crops, and brightness and contrast changes, to imitate the variety of phone photos taken in the field.

In [ ]:
def stratified_split(files_by_class, val_frac=0.15, test_frac=0.15, seed=SEED):
    rng = random.Random(seed)
    splits = {"train": [], "val": [], "test": []}
    for label, c in enumerate(CLASSES):
        files = list(files_by_class[c])
        rng.shuffle(files)
        n_val = max(1, round(len(files) * val_frac))
        n_test = max(1, round(len(files) * test_frac))
        splits["val"] += [(f, label) for f in files[:n_val]]
        splits["test"] += [(f, label) for f in files[n_val:n_val + n_test]]
        splits["train"] += [(f, label) for f in files[n_val + n_test:]]
    for items in splits.values():
        rng.shuffle(items)
    return splits

splits = stratified_split(files_by_class)
for name, items in splits.items():
    print(f"{name:5s} {len(items):6d} images")

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label):
    image = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))   # float32 in [0, 255]
    return image, label

def augment(image, label):
    # Zoom by upscaling a little and cropping back to size.
    image = tf.image.resize(image, (IMG_SIZE + 16, IMG_SIZE + 16))
    image = tf.image.random_crop(image, (IMG_SIZE, IMG_SIZE, 3))
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.rot90(image, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
    image = tf.image.random_brightness(image, max_delta=30.0)
    image = tf.image.random_contrast(image, 0.8, 1.2)
    return tf.clip_by_value(image, 0.0, 255.0), label

def scale(image, label):
    # Identical to convertBitmapToByteBuffer in TFLiteClassifier.java.
    return image / 127.5 - 1.0, label

def make_dataset(items, training):
    paths = [p for p, _ in items]
    labels = [l for _, l in items]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(len(items), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.map(scale, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(splits["train"], training=True)
val_ds = make_dataset(splits["val"], training=False)
test_ds = make_dataset(splits["test"], training=False)

In [ ]:
# Preview a few augmented training images.
images, labels = next(iter(train_ds))
plt.figure(figsize=(12, 6))
for i in range(min(8, len(images))):
    plt.subplot(2, 4, i + 1)
    plt.imshow((images[i].numpy() + 1.0) / 2.0)
    plt.title(CLASSES[int(labels[i])].replace("___", "\n").replace("_", " "), fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

The classes are very uneven (tomato yellow leaf curl virus has over 5,000 photos, healthy potato about 150), so each class's loss is weighted by the inverse of its size. Without this the model learns to under-predict the rare classes.

In [ ]:
train_counts = np.bincount([l for _, l in splits["train"]], minlength=len(CLASSES))
class_weight = {i: len(splits["train"]) / (len(CLASSES) * n) for i, n in enumerate(train_counts)}
for i, c in enumerate(CLASSES):
    print(f"{class_weight[i]:6.2f}  {c}")

## 4. Build the model

MobileNetV2 pretrained on ImageNet, with a new softmax layer for our classes. The backbone's own preprocessing is left out on purpose: the app scales pixels to [-1, 1] before calling the model.

In [ ]:
base = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), alpha=ALPHA, include_top=False, weights="imagenet")
base.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
# training=False keeps BatchNormalization in inference mode, also during fine tuning.
x = base(inputs, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(len(CLASSES), activation="softmax", name="probabilities")(x)
model = keras.Model(inputs, outputs, name="farmroute_disease")
model.summary()

## 5. Train

Stage 1 trains only the new head while the pretrained backbone stays frozen. Stage 2 unfreezes the top of the backbone and continues with a much lower learning rate, so it adapts to leaves without forgetting what it learned from ImageNet. The best checkpoint by validation accuracy is kept.

In [ ]:
CHECKPOINT = os.path.join(WORK_DIR, "best.keras")

def callbacks():
    return [
        keras.callbacks.ModelCheckpoint(CHECKPOINT, monitor="val_accuracy", save_best_only=True),
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    ]

model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_head = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
                         class_weight=class_weight, callbacks=callbacks())

In [ ]:
base.trainable = True
for layer in base.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(2e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history_fine = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE_TUNE,
                         class_weight=class_weight, callbacks=callbacks())

In [ ]:
def curve(key):
    return history_head.history[key] + history_fine.history[key]

plt.figure(figsize=(12, 4))
for i, metric in enumerate(["accuracy", "loss"]):
    plt.subplot(1, 2, i + 1)
    plt.plot(curve(metric), label="train")
    plt.plot(curve("val_" + metric), label="validation")
    plt.axvline(len(history_head.history[metric]) - 0.5, color="grey", linestyle="--")
    plt.title(metric + " (dashed line: start of fine tuning)")
    plt.xlabel("epoch")
    plt.legend()
plt.show()

## 6. Evaluate on the held out test set

These images were never used for training or for choosing the checkpoint. PlantVillage photos are taken on plain backgrounds, so expect accuracy on real field photos to be lower than this number; that gap is what roadmap phase 5 (retraining on real captures) is for.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
print(f"Keras model test accuracy: {test_accuracy:.4f}\n")

y_true = np.array([l for _, l in splits["test"]])
y_prob = model.predict(test_ds, verbose=0)
y_pred = y_prob.argmax(axis=1)
short_names = [c.replace("Corn_(maize)", "Maize").replace("Pepper,_bell", "Pepper")
               .replace("___", ": ").replace("_", " ") for c in CLASSES]
print(classification_report(y_true, y_pred, labels=range(len(CLASSES)),
                            target_names=short_names, digits=3, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)), normalize="true")
plt.figure(figsize=(11, 9))
plt.imshow(cm, cmap="Greens", vmin=0, vmax=1)
plt.xticks(range(len(CLASSES)), short_names, rotation=90, fontsize=8)
plt.yticks(range(len(CLASSES)), short_names, fontsize=8)
plt.xlabel("predicted")
plt.ylabel("true")
plt.colorbar(label="fraction of true class")
plt.title("Confusion matrix (test set)")
plt.tight_layout()
plt.show()

## 7. Export to TensorFlow Lite

Full integer (int8) quantisation shrinks the model about four times and speeds it up on phone CPUs. A few hundred training images are used to calibrate the value ranges. The input and output stay float32, so the app's code needs no change.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
saved_model_dir = os.path.join(WORK_DIR, "farmroute_saved_model")
try:
    model.export(saved_model_dir, verbose=False)
except TypeError:  # Older Keras versions have no verbose argument.
    model.export(saved_model_dir)

calibration_items = random.Random(SEED).sample(splits["train"], min(300, len(splits["train"])))

def representative_dataset():
    for path, label in calibration_items:
        image, _ = scale(*load_image(path, label))
        yield [image[tf.newaxis, ...]]

converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
tflite_model = converter.convert()

model_path = os.path.join(OUTPUT_DIR, MODEL_FILE)
with open(model_path, "wb") as f:
    f.write(tflite_model)
labels_path = os.path.join(OUTPUT_DIR, LABELS_FILE)
with open(labels_path, "w") as f:
    f.write("\n".join(CLASSES) + "\n")

print(f"Wrote {model_path} ({len(tflite_model) / 1e6:.2f} MB)")
print(f"Wrote {labels_path}")

### Check the exported model against the app's expectations

Loads the `.tflite` file with the TensorFlow Lite interpreter, confirms the input and output match what `TFLiteClassifier.java` sends and reads, then measures its accuracy on the test set one image at a time, as the app runs it. The quantised model should be within a percentage point or two of the Keras model.

In [ ]:
try:
    # tf.lite.Interpreter is deprecated in favour of LiteRT and will be removed.
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    Interpreter = tf.lite.Interpreter

interpreter = Interpreter(model_path=model_path)
interpreter.allocate_tensors()
input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]
print("input :", input_detail["shape"], input_detail["dtype"].__name__)
print("output:", output_detail["shape"], output_detail["dtype"].__name__)

assert list(input_detail["shape"]) == [1, IMG_SIZE, IMG_SIZE, 3], "input shape does not match the app"
assert input_detail["dtype"] == np.float32, "app sends float32 pixels"
assert list(output_detail["shape"]) == [1, len(CLASSES)], "output size must equal the number of labels"
assert output_detail["dtype"] == np.float32, "app reads float32 probabilities"

correct = 0
for path, label in splits["test"]:
    image, _ = scale(*load_image(path, label))
    interpreter.set_tensor(input_detail["index"], image.numpy()[np.newaxis, ...])
    interpreter.invoke()
    correct += int(interpreter.get_tensor(output_detail["index"])[0].argmax() == label)
tflite_accuracy = correct / len(splits["test"])
print(f"\nKeras test accuracy : {test_accuracy:.4f}")
print(f"TFLite test accuracy: {tflite_accuracy:.4f}")

## 8. Download and install in the app

Downloads both files to your computer (in Colab). Then:

1. Copy `farmroute_disease_model.tflite` and `labels.txt` into `app/src/main/assets/`, replacing the placeholder `labels.txt`.
2. Rebuild the app in Android Studio.

The `.gitignore` keeps the model file out of git by default. Remove that line if you want to commit the model.

In [ ]:
try:
    from google.colab import files
    files.download(model_path)
    files.download(labels_path)
except ImportError:
    print("Not running in Colab. Files are in", OUTPUT_DIR)